# Unidad 2.1 - smolagents

Este notebook cubre los conceptos principales del framework **smolagents** de Hugging Face para construir agentes de IA.

**Temas cubiertos:**
1. CodeAgents — agentes que escriben y ejecutan código Python
2. ToolCallingAgents — agentes que usan JSON para llamar herramientas
3. Retrieval Agents — agentes con búsqueda web y base de conocimiento personalizada

> Alfred, el mayordomo de Wayne Manor, es nuestro agente protagonista en esta unidad. 🦇

## Instalación y configuración

In [1]:
%pip install smolagents -U
%pip install ddgs
%pip install langchain-community langchain-text-splitters rank_bm25
%pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from huggingface_hub import login

# Inicia sesión en Hugging Face Hub para acceder a la Serverless Inference API.
# Si tienes el token como variable de entorno (HF_TOKEN) ya estás autenticado.
# Alternativa sin widget: login(token="hf_tu_token_aquí")
try:
    login()
except ImportError:
    # ipywidgets no disponible — usa token directamente
    import os
    token = os.environ.get("HF_TOKEN", "")
    if token:
        login(token=token)
    else:
        print("Instala ipywidgets o define la variable de entorno HF_TOKEN.")
        print("Ejemplo: set HF_TOKEN=hf_tu_token  (Windows)")
        print("         $env:HF_TOKEN='hf_tu_token'  (PowerShell)")

---
## Parte 1: CodeAgents

Los **CodeAgents** son el tipo de agente principal en `smolagents`. En lugar de generar JSON o texto, estos agentes producen **código Python** para realizar acciones.

**Ventajas del código sobre JSON:**
- **Composabilidad**: combinación y reutilización fácil de acciones
- **Gestión de objetos**: trabajo directo con estructuras complejas como imágenes
- **Generalidad**: permite expresar cualquier tarea computacionalmente posible
- **Natural para LLMs**: el código de alta calidad ya está presente en los datos de entrenamiento

### Ejemplo 1: Seleccionar una lista de reproducción para la fiesta

In [3]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, InferenceClientModel

# Creamos un agente con la herramienta de búsqueda DuckDuckGo
# El modelo predeterminado es Qwen/Qwen2.5-Coder-32B-Instruct
agent = CodeAgent(tools=[DuckDuckGoSearchTool()], model=InferenceClientModel())

agent.run("Busca las mejores recomendaciones musicales para una fiesta en la mansión Wayne.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Busca las mejores recomendaciones musicales para una fiesta en la mansión Wayne.                                │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: 
Root=1-6a0d13ad-64bb676809932df62e4aa195;fda94f12-e304-4729-a3e2-4774eb94ad88)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. 
Alternatively, subscribe to PRO to get 20x more included usage.

[Step 1: Duration 0.57 seconds]

AgentGenerationError: Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a0d13ad-64bb676809932df62e4aa195;fda94f12-e304-4729-a3e2-4774eb94ad88)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

### Ejemplo 2: Herramienta personalizada para preparar el menú

Usamos el decorador `@tool` para definir una función personalizada que actúa como herramienta.

In [4]:
from smolagents import CodeAgent, tool, InferenceClientModel

@tool
def suggest_menu(occasion: str) -> str:
    """
    Sugiere un menú basado en el tipo de ocasión.
    Args:
        occasion (str): El tipo de ocasión para la fiesta. Valores permitidos:
                        - "casual": Menú para fiesta informal.
                        - "formal": Menú para fiesta formal.
                        - "superhero": Menú para fiesta de superhéroes.
                        - "custom": Menú personalizado.
    """
    if occasion == "casual":
        return "Pizza, snacks y bebidas."
    elif occasion == "formal":
        return "Cena de 3 platos con vino y postre."
    elif occasion == "superhero":
        return "Buffet con comida energética y saludable."
    else:
        return "Menú personalizado para el mayordomo."

# Alfred prepara el menú para la fiesta
agent = CodeAgent(tools=[suggest_menu], model=InferenceClientModel())

agent.run("Prepara un menú formal para la fiesta.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Prepara un menú formal para la fiesta.                                                                          │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: 
Root=1-6a0d13ae-3f10b52f3936623457369ee5;a7e180ab-b2e6-4f56-8a9c-6d5a98d77b6c)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. 
Alternatively, subscribe to PRO to get 20x more included usage.

[Step 1: Duration 0.11 seconds]

AgentGenerationError: Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a0d13ae-3f10b52f3936623457369ee5;a7e180ab-b2e6-4f56-8a9c-6d5a98d77b6c)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

### Ejemplo 3: Importaciones Python dentro del agente

`smolagents` ejecuta código en un entorno seguro. Las importaciones fuera de una lista predefinida están bloqueadas, pero se pueden autorizar con `additional_authorized_imports`.

In [5]:
from smolagents import CodeAgent, InferenceClientModel

agent = CodeAgent(
    tools=[],
    model=InferenceClientModel(),
    additional_authorized_imports=['datetime']
)

agent.run(
    """
    Alfred necesita preparar la fiesta. Estas son las tareas:
    1. Preparar las bebidas - 30 minutos
    2. Decorar la mansión - 60 minutos
    3. Preparar el menú - 45 minutos
    4. Preparar la música y la lista de reproducción - 45 minutos

    Si comenzamos ahora mismo, ¿a qué hora estará lista la fiesta?
    """
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Alfred necesita preparar la fiesta. Estas son las tareas:                                                       │
│     1. Preparar las bebidas - 30 minutos                                                                        │
│     2. Decorar la mansión - 60 minutos                                                                          │
│     3. Preparar el menú - 45 minutos                                                                            │
│     4. Preparar la música y la lista de reproducción - 45 minutos                                               │
│                                                                                                                 │
│     Si comenzamos ahora mismo, ¿a qué hora estará lista la fiesta?                                              │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: 
Root=1-6a0d13ae-2dec7f4a7bf6869625d7aa55;df763c53-8f5c-407a-9a05-1f437e6f80b2)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. 
Alternatively, subscribe to PRO to get 20x more included usage.

[Step 1: Duration 0.10 seconds]

AgentGenerationError: Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a0d13ae-2dec7f4a7bf6869625d7aa55;df763c53-8f5c-407a-9a05-1f437e6f80b2)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

### Ejemplo 4: Agente completo para planificar la fiesta

Combinamos múltiples herramientas para crear el agente "Alfred" definitivo.

In [6]:
from smolagents import (
    CodeAgent, DuckDuckGoSearchTool, FinalAnswerTool,
    InferenceClientModel, Tool, tool, VisitWebpageTool
)

@tool
def suggest_menu(occasion: str) -> str:
    """
    Sugiere un menú basado en el tipo de ocasión.
    Args:
        occasion: El tipo de ocasión para la fiesta.
    """
    if occasion == "casual":
        return "Pizza, snacks y bebidas."
    elif occasion == "formal":
        return "Cena de 3 platos con vino y postre."
    elif occasion == "superhero":
        return "Buffet con comida energética y saludable."
    else:
        return "Menú personalizado para el mayordomo."

@tool
def catering_service_tool(query: str) -> str:
    """
    Devuelve el servicio de catering mejor valorado en Ciudad Gótica.
    Args:
        query: Término de búsqueda para encontrar servicios de catering.
    """
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }
    best_service = max(services, key=services.get)
    return best_service

class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = """
    Sugiere ideas creativas para fiestas temáticas de superhéroes según una categoría.
    Devuelve una idea única para el tema de la fiesta."""

    inputs = {
        "category": {
            "type": "string",
            "description": "El tipo de fiesta de superhéroes (ej: 'classic heroes', 'villain masquerade', 'futuristic Gotham').",
        }
    }
    output_type = "string"

    def forward(self, category: str):
        themes = {
            "classic heroes": "Gala de la Liga de la Justicia: Los invitados vienen disfrazados de sus héroes favoritos de DC.",
            "villain masquerade": "Baile de máscaras de los Rogues de Gotham: Una mascarada misteriosa con villanos clásicos de Batman.",
            "futuristic gotham": "Neo-Gotham Night: Fiesta cyberpunk inspirada en Batman Beyond, con decoraciones de neón."
        }
        return themes.get(category.lower(), "Tema no encontrado. Prueba 'classic heroes', 'villain masquerade', o 'futuristic Gotham'.")

# Alfred, el mayordomo, con todas sus herramientas
agent = CodeAgent(
    tools=[
        DuckDuckGoSearchTool(),
        VisitWebpageTool(),
        suggest_menu,
        catering_service_tool,
        SuperheroPartyThemeTool(),
        FinalAnswerTool()
    ],
    model=InferenceClientModel(),
    max_steps=10,
    verbosity_level=2
)

agent.run("Dame la mejor lista de reproducción para una fiesta en la mansión Wayne. El tema es una 'mascarada de villanos'.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Dame la mejor lista de reproducción para una fiesta en la mansión Wayne. El tema es una 'mascarada de           │
│ villanos'.                                                                                                      │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: 
Root=1-6a0d13af-078449907382e5b571107598;93e9ed56-6a6b-4bbe-87bd-950052eb280e)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. 
Alternatively, subscribe to PRO to get 20x more included usage.

[Step 1: Duration 0.13 seconds]

AgentGenerationError: Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a0d13af-078449907382e5b571107598;93e9ed56-6a6b-4bbe-87bd-950052eb280e)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

---
## Parte 2: ToolCallingAgents

Los **ToolCallingAgents** son el segundo tipo de agente en `smolagents`. A diferencia de los CodeAgents que usan Python, estos agentes usan las **capacidades nativas de llamada a herramientas** de los proveedores de LLM para generar estructuras **JSON**.

### Comparación: CodeAgent vs ToolCallingAgent

**CodeAgent** genera y ejecuta código Python:
```python
for query in ["Mejores servicios de catering en Gotham", "Ideas para fiestas de superhéroes"]:
    print(web_search(f"Búsqueda: {query}"))
```

**ToolCallingAgent** genera una estructura JSON:
```python
[
    {"name": "web_search", "arguments": "Mejores servicios de catering en Gotham"},
    {"name": "web_search", "arguments": "Ideas para fiestas de superhéroes"}
]
```

In [7]:
from smolagents import ToolCallingAgent, DuckDuckGoSearchTool, InferenceClientModel

# ToolCallingAgent requiere un modelo con soporte nativo de tool-calling (function calling)
# Usamos Qwen2.5-72B-Instruct que sí lo soporta vía HF router
try:
    agent = ToolCallingAgent(
        tools=[DuckDuckGoSearchTool()],
        model=InferenceClientModel(model_id="Qwen/Qwen2.5-72B-Instruct")
    )
    result = agent.run("Busca las mejores recomendaciones musicales para una fiesta en la mansión Wayne.")
    print(result)
except Exception as e:
    print(f"⚠️  ToolCallingAgent: {e}")
    print("\nNota: ToolCallingAgent genera estructuras JSON (no código Python) usando las capacidades")
    print("nativas de tool-calling del modelo. Requiere un proveedor compatible con function calling.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Busca las mejores recomendaciones musicales para una fiesta en la mansión Wayne.                                │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error while generating output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: 
Root=1-6a0d13af-49a6bbc35dfa98f367f252a2;9f3cd4c4-aa3c-4d44-8e60-d6657b57f6ad)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. 
Alternatively, subscribe to PRO to get 20x more included usage.

[Step 1: Duration 0.11 seconds]

⚠️  ToolCallingAgent: Error while generating output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a0d13af-49a6bbc35dfa98f367f252a2;9f3cd4c4-aa3c-4d44-8e60-d6657b57f6ad)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

Nota: ToolCallingAgent genera estructuras JSON (no código Python) usando las capacidades
nativas de tool-calling del modelo. Requiere un proveedor compatible con function calling.


**Nota:** En la traza del agente verás `Calling tool: 'web_search' with arguments: {...}` en lugar de `Executing parsed code:`, que es la diferencia principal entre ambos tipos.

---
## Parte 3: Retrieval Agents (RAG Agéntico)

Los sistemas **RAG (Retrieval-Augmented Generation)** combinan recuperación de información con generación para dar respuestas conscientes del contexto.

El RAG agéntico extiende esto permitiendo al agente:
- Formular consultas de búsqueda autónomamente
- Criticar los resultados recuperados
- Realizar múltiples pasos de recuperación

### Ejemplo 1: Recuperación básica con DuckDuckGo

In [8]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, InferenceClientModel

search_tool = DuckDuckGoSearchTool()
model = InferenceClientModel()

agent = CodeAgent(
    model=model,
    tools=[search_tool],
)

response = agent.run(
    "Busca ideas de fiesta de superhéroes de lujo, incluyendo decoraciones, entretenimiento y catering."
)
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Busca ideas de fiesta de superhéroes de lujo, incluyendo decoraciones, entretenimiento y catering.              │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: 
Root=1-6a0d13af-6668be871a74234233bd5e64;81a50c5c-3135-4b2d-92a1-15879628385c)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. 
Alternatively, subscribe to PRO to get 20x more included usage.

[Step 1: Duration 0.10 seconds]

AgentGenerationError: Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a0d13af-6668be871a74234233bd5e64;81a50c5c-3135-4b2d-92a1-15879628385c)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

### Ejemplo 2: Herramienta de base de conocimiento personalizada

Para tareas especializadas, una base de conocimiento personalizada es invaluable. Usamos un recuperador **BM25** para búsqueda semántica sobre documentos propios.

---
## Resumen

| Concepto | Descripción |
|---|---|
| **CodeAgent** | Genera y ejecuta código Python para realizar acciones |
| **ToolCallingAgent** | Genera estructuras JSON para llamar herramientas |
| **@tool decorator** | Forma simple de crear herramientas desde funciones Python |
| **Tool class** | Forma avanzada de crear herramientas con control total |
| **RAG agéntico** | El agente controla el proceso de recuperación y generación |
| **additional_authorized_imports** | Permite importar módulos adicionales en el sandbox |

**Recursos:**
- [Documentación oficial de smolagents](https://huggingface.co/docs/smolagents)
- [Ejecutable Code Actions Elicit Better LLM Agents](https://huggingface.co/papers/2402.01030)
- [Guía de RAG agéntico](https://huggingface.co/learn/cookbook/agent_rag)

In [9]:
from langchain_community.docstore.document import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from smolagents import Tool
from langchain_community.retrievers import BM25Retriever
from smolagents import CodeAgent, InferenceClientModel

class PartyPlanningRetrieverTool(Tool):
    name = "party_planning_retriever"
    description = "Usa búsqueda semántica para recuperar ideas relevantes de planificación de fiestas para la fiesta de superhéroes de Alfred en Wayne Manor."
    inputs = {
        "query": {
            "type": "string",
            "description": "La consulta a realizar. Debe estar relacionada con planificación de fiestas o temáticas de superhéroes.",
        }
    }
    output_type = "string"

    def __init__(self, docs, **kwargs):
        super().__init__(**kwargs)
        self.retriever = BM25Retriever.from_documents(
            docs, k=5
        )

    def forward(self, query: str) -> str:
        assert isinstance(query, str), "La consulta debe ser una cadena de texto"
        docs = self.retriever.invoke(query)
        return "\nIdeas recuperadas:\n" + "".join(
            [
                f"\n\n===== Idea {str(i)} =====\n" + doc.page_content
                for i, doc in enumerate(docs)
            ]
        )

# Base de conocimiento simulada sobre planificación de fiestas
party_ideas = [
    {"text": "Un baile de máscaras con temática de superhéroes con decoración de lujo, incluyendo acentos dorados y cortinas de terciopelo.", "source": "Ideas de Fiesta 1"},
    {"text": "Contratar un DJ profesional que pueda poner música temática de superhéroes como Batman y Wonder Woman.", "source": "Ideas de Entretenimiento"},
    {"text": "Para el catering, sirve platos con nombre de superhéroes, como 'El Batido Verde del Hulk' y 'El Filete de Poder de Iron Man'.", "source": "Ideas de Catering"},
    {"text": "Decora con los logos icónicos de superhéroes y proyecciones de Gotham y otras ciudades de superhéroes alrededor del venue.", "source": "Ideas de Decoración"},
    {"text": "Experiencias interactivas con VR donde los invitados puedan participar en simulaciones de superhéroes o competir en juegos temáticos.", "source": "Ideas de Entretenimiento"}
]

source_docs = [
    Document(page_content=doc["text"], metadata={"source": doc["source"]})
    for doc in party_ideas
]

# Dividir los documentos en fragmentos más pequeños para búsqueda más eficiente
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True,
    strip_whitespace=True,
    separators=["\n\n", "\n", ".", " ", ""],
)
docs_processed = text_splitter.split_documents(source_docs)

# Crear la herramienta de recuperación
party_planning_retriever = PartyPlanningRetrieverTool(docs_processed)

# Inicializar el agente
agent = CodeAgent(tools=[party_planning_retriever], model=InferenceClientModel())

response = agent.run(
    "Encuentra ideas para una fiesta de superhéroes de lujo, incluyendo opciones de entretenimiento, catering y decoración."
)

print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Encuentra ideas para una fiesta de superhéroes de lujo, incluyendo opciones de entretenimiento, catering y      │
│ decoración.                                                                                                     │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: 
Root=1-6a0d13b8-4a8b29b42ad623f8674a5d93;b9a07be3-f4f9-4398-9319-c909e4b83bff)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. 
Alternatively, subscribe to PRO to get 20x more included usage.

[Step 1: Duration 0.14 seconds]

AgentGenerationError: Error in generating model output:
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a0d13b8-4a8b29b42ad623f8674a5d93;b9a07be3-f4f9-4398-9319-c909e4b83bff)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.